# Cakap — Fine-tuning Gemma 4 for Indonesian Customer Service

Fine-tuning Gemma 4 (4B) using Unsloth on Google Colab for Indonesian-language customer service conversations.

**Dataset:** [Indonesian CS Conversation Dataset](https://www.kaggle.com/datasets/rafikusuma/cs-dataset)  
**Model:** [rafkus/gemma4-cs-q8_0](https://huggingface.co/rafkus/gemma4-cs-q8_0)  

---

## 1. Setup

In [ ]:
# Clear cache (run this if you hit memory issues)
!rm -rf /root/.cache/huggingface/hub/*
!rm -rf /root/.cache/pip/*
!rm -rf /content/outputs/checkpoint-*

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers accelerate bitsandbytes peft
!pip install -q "trl>=0.18.2,<=0.24.0" "datasets>=3.4.1,<4.4.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load Model

In [ ]:
from unsloth import FastVisionModel
import torch

max_seq_length = 2048
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    load_in_4bit = True,
    device_map = "auto",
    dtype = dtype,
)

# Apply LoRA — fine-tune language layers only (not vision)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Model loaded successfully.")

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset
from unsloth import get_chat_template

# Apply Gemma chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma")

# Load dataset from Google Drive
# Download from: https://www.kaggle.com/datasets/rafikusuma/cs-dataset
file_path = "/content/drive/MyDrive/dataset_cs_text.json"
dataset = load_dataset("json", data_files=file_path, split="train")

def format_dataset(examples):
    texts = []
    for convo in examples["conversations"]:
        formatted_convo = []
        system_prompt = ""

        for message in convo:
            role = message["role"]
            content = message["content"]

            # Merge system prompt into first user message (Gemma doesn't support system role)
            if role == "system":
                system_prompt = content + "\n\n"
                continue

            if role == "user" and system_prompt:
                content = system_prompt + content
                system_prompt = ""

            if role == "model":
                role = "assistant"

            formatted_convo.append({"role": role, "content": content})

        text = tokenizer.apply_chat_template(formatted_convo, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(format_dataset, batched=True)

print(f"Dataset ready: {len(dataset)} conversations")
print("\nSample:")
print(dataset[0]["text"][:500])

## 4. Fine-tune

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()
print("Training complete.")

## 5. Export to GGUF

In [ ]:
import os

model_path = os.path.join(os.getcwd(), "gemma4_cs_finetuned")
model.save_pretrained_gguf(model_path, tokenizer, quantization_method="q4_k_m")

print(f"Model saved to: {model_path}")

In [ ]:
# Save to Google Drive
!zip -r model_cakap.zip gemma4_cs_finetuned/
!cp model_cakap.zip /content/drive/MyDrive/
print("Saved to Google Drive: model_cakap.zip")